# Running Kinessence: the essentials

How to build a model, run the solver, check it, compare it to ΛCDM, compute
the growth and save the result.

In [1]:
import sys
sys.path.insert(0, "..")

from kmouflage import (
    CosmologicalParams, InitialConditions,
    make_powerlaw_K, make_arctan_K, make_exponential_coupling, make_exponential_potential,
    make_LambdaCDM_K, make_LambdaCDM_coupling,
    run_background, verify, calibrate_M4_tilde,
    summary_table, print_summary_table,
    KMouflageGrowth,
    save_table, save_pysco_table,
)

## 1. Cosmology & initial conditions

In [2]:
cosmo = CosmologicalParams(
    omega_m=0.1430,       # Omega_m0 * h^2 (Planck 2018)
    omega_r=4.15e-5,      # Omega_r0 * h^2 (Planck 2018)
    H100=100.0,           # fixed reference unit [km/s/Mpc]
    M_Pl_input=2.435e18,  # reduced Planck mass [GeV]
    M4_tilde=1.0,         # input of the model, H0 is then a prediction
)
ic = InitialConditions(
    z_ini=1e8,            # where the integration starts, 1e8 is needed for the pysco table (section 8)
    phi_ini=0.0,          # initial scalar field value
)

print(cosmo)
print(ic)

CosmologicalParams(H100=100.0, Mpl=1.0, M_Pl_input=2.435e+18, omega_m=0.143, omega_r=4.15e-05, M4_tilde=1.0)
InitialConditions(z_ini=100000000.0, phi_ini=0.0)


## 2. Choose K(X), the coupling A(φ), and the potential f(φ)

model = K(X) kinetic term, coupling = A(φ) conformal coupling to matter, potential = f(φ). Leaving potential=None (the default) is bare K-mouflage (f≡1), adding potential f ≠ 1 , like below, is what this project calls a Kinessence model.

All three are built with make_* factories, see `custom_models_guide.ipynb` to write your own instead.

In [3]:
model     = make_powerlaw_K(K0=1, m=3)             # K(X) = X + K0*X^m
coupling  = make_exponential_coupling(beta=0.1)    # A(phi) = exp(beta*phi)
potential = make_exponential_potential(lam=1.0)    # f(phi) = exp(-lam*phi) -> Kinessence

## 3. Run it

run_background builds a KMouflageBackground and calls bg.run(). The object then exposes every derived quantity as a function of N = ln(a) (bg.phi(N), bg.Omega_m(N), ...). Passing potential=potential is what makes this a Kinessence run instead of bare K-mouflage.

In [4]:
bg = run_background(model, coupling, potential=potential, cosmo=cosmo, ic=ic, verbose=True)

print(f"H0_predicted        = {bg.H0_predicted:.4f} km/s/Mpc")
print(f"Omega_m0_predicted  = {bg.Omega_m0_predicted:.4f}")
print(f"Omega_de0_predicted = {bg.Omega_de0_predicted:.4f}")
print(f"phi(z=0)            = {bg.phi(0.0):.4f}")


[RUN] z_ini=1.0e+08 | omega_m=0.143 | omega_r=4.15e-05


[OK] φ̃(z=0)=-7.4575e-02 | M_Pl0_eff=1.007485 | delta_Mpl=+7.4854e-03
[PREDICTED] h=0.766578 | H0=76.6578 km/s/Mpc | Omega_m0=0.2433 | Omega_de0=0.7566
[CHECK] Omega_DE(z_ini=1.0e+08) = -1.665e-13 (should be << 1 for the deep-RD omega_m/omega_r extrapolation to be valid)
H0_predicted        = 76.6578 km/s/Mpc
Omega_m0_predicted  = 0.2433
Omega_de0_predicted = 0.7566
phi(z=0)            = -0.0746


## 3bis. Any physical quantity, bg.<name>(N)

In [5]:
N = 0.0   # N = ln(a); any N, or a numpy array of N, works

print("rho_m         =", bg.rho_m(N))          # matter density
print("rho_r         =", bg.rho_r(N))          # radiation density
print("rho_phi       =", bg.rho_phi(N))        # scalar field density
print("p_phi         =", bg.p_phi(N))          # scalar field pressure
print("w_phi         =", bg.w_phi(N))          # scalar field equation of state
print("w_de_def      =", bg.w_de_def(N))       # dark energy equation of state
print("mu_K          =", bg.mu_K(N))           # effective Newton constant
print("epsilon_F     =", bg.epsilon_F(N))      # BBN / G_eff constraint quantity
print("Z             =", bg.Z(N))              # kinetic normalization
print("Z_eff         =", bg.Z_eff(N))          # kinetic normalization, no-ghost condition
print("F             =", bg.F(N))              # (M_Pl,eff / M_Pl)^2
print("H             =", bg.H(N))              # physical Hubble rate
print("t_cosmic      =", bg.t_cosmic(N))       # cosmic time

# full list: see KMouflageBackground._build_interpolators in solver.py

rho_m         = 0.42899999999999994
rho_r         = 0.0001245
rho_phi       = 1.181750082492704
p_phi         = -1.0398908036669152
w_phi         = -0.8799583085058387
w_de_def      = -0.8776430659352015
mu_K          = 1.0045974776824462
epsilon_F     = -0.014804314331935742
Z             = 0.9699310085833691
Z_eff         = 1.0331543672664727
F             = 1.015026775438929
H             = 0.7665782941167291
t_cosmic      = 1.3390879526246073


## 4. Sanity-check, verify

verify(bg) checks the run itself (numerical derivatives against the equations,
Friedmann, closure of the Omegas), the stability of the field along the
trajectory, and the Solar System tests (Cassini and the Moon periastron) with
the M4_tilde of the run. The power law model fails the Moon test, that is
expected for this family of models.

In [6]:
verify(bg)

verify: power-law (K0=1, m=3), exponential (β=0.1), exponential (lam=1.0), M4_tilde=1
  check                                value     bound  status
  ──────────────────────────────────────────────────────────────────────
  dphi/dN  num vs phi'/H           1.777e-08   1.0e-03  PASS  
  dphi'/dN num vs phi''/H          1.176e-08   1.0e-03  PASS  
  dlnH/dN  num vs H'/H^2           9.009e-09   1.0e-03  PASS  Friedmann 2 vs derivative of Friedmann 1
  Friedmann 1 residual             9.639e-16   1.0e-03  PASS  
  Omega_m+Omega_r+Omega_DE-1       1.110e-15   1.0e-03  PASS  
  Omega_DE at z_ini               -1.665e-13   1.0e-03  PASS  initial conditions assume no dark energy
  min K'                           1.000e+00            PASS  no ghost
  min K' + 2X K''                  1.000e+00            PASS  demixed KG regular
  min c_s^2                        2.000e-01            PASS  max c_s^2 = 1.000
  epsilon_F today                 -1.480e-02                  M_Pl^2/F - 1
  X at 1 AU  

## 5. Comparing models

ΛCDM is run with the same cosmology and the same M4_tilde, so Λ = M4_tilde:
no coupling and a constant potential. The table shows how far each model goes
from it.

In [7]:
bg_arctan = run_background(make_arctan_K(K_star=1000, X_star=100), coupling, potential=potential, cosmo=cosmo, ic=ic)
bg_lcdm   = run_background(make_LambdaCDM_K(), make_LambdaCDM_coupling(), cosmo=cosmo, ic=ic)

rows = summary_table(
    {"power-law": bg, "arctan": bg_arctan, "ΛCDM": bg_lcdm},
    ref=bg_lcdm,
)
print_summary_table(rows)

model                    H0    ΔH0(%)       Ω_m0   ΔΩ_m0(%)      Ω_DE0   ΔΩ_DE0(%)      w_DE0   Δw_DE0(%)
─────────────────────────────────────────────────────────────────────────────────────────────────────────
power-law           76.6578   +11.066    0.24335    -18.935    0.75658      +8.125   -0.87764     -12.236
arctan              79.5584   +15.269    0.22592    -24.738    0.77401     +10.616   -0.86761     -13.239
ΛCDM                69.0199    +0.000    0.30018     +0.000    0.69973      +0.000   -1.00000      -0.000


## 6. Calibrating M4_tilde (optional)

By default we don't calibrate: M4_tilde is an input and H0 comes out as a
prediction. If you still want a model that hits a given H0, calibrate_M4_tilde
tunes M4_tilde for you. It changes the background it is given, so use a
separate run.

In [8]:
bg_cal = run_background(model, coupling, potential=potential, cosmo=cosmo, ic=ic)
M4_tilde, H0_final, n_iter = calibrate_M4_tilde(bg_cal, target_H0=67.36, verbose=False)
print(f"M4_tilde = {M4_tilde:.6f}  ->  H0_predicted = {bg_cal.H0_predicted:.4f} km/s/Mpc  ({n_iter} iterations)")

M4_tilde = 0.671436  ->  H0_predicted = 67.3600 km/s/Mpc  (10 iterations)


## 7. Linear growth

KMouflageGrowth(bg) gives the growth rate f = dlnD+/dlna and the growth
factor D+ (equal to 1 today) from a background that has already run. Running
it on the ΛCDM background gives the reference.

In [9]:
growth      = KMouflageGrowth(bg, N_points=2000).run(verbose=False)
growth_lcdm = KMouflageGrowth(bg_lcdm, N_points=2000).run(verbose=False)

print(f"f(z=0)  : model={growth['f'][-1]:.5f}   ΛCDM={growth_lcdm['f'][-1]:.5f}")
print(f"D+(z=0) : model={growth['Dp'][-1]:.5f}   ΛCDM={growth_lcdm['Dp'][-1]:.5f}")

f(z=0)  : model=0.44958   ΛCDM=0.51296
D+(z=0) : model=1.00000   ΛCDM=1.00000


## 8. Saving the result

save_table writes one CSV with a commented header (model, coupling, potential,
cosmology) and the fields you ask for as columns.

save_pysco_table writes the table pysco reads: aexp, E = H/H0, mu_K, Omega_m,
Omega_r and py_DE, from a = 1e-8 to today. That is why this notebook starts at
z_ini = 1e8.

In [10]:
from kmouflage.run import DEFAULT_FIELDS
print(DEFAULT_FIELDS)   # saved by default if fields= is omitted

path = save_table(bg, outdir="runs")
# path = save_table(bg, outdir="runs", fields=["Z", "mu_K", "rho_phi", "p_phi"])   # only the fields you want
print(path)

with open(path) as f:
    print("".join(f.readlines()[:7]))

pysco_path = save_pysco_table(bg, path="runs/pysco_table.txt", overwrite=True)
print(pysco_path)

['z', 'a', 'phi', 'phi_prime', 't_cosmic', 'eta', 't_superconform', 'H', 'H_conf', 'Omega_m', 'Omega_r', 'Omega_de_def', 'w_de_def', 'w_phi']


runs/kmouflage_table_exponential_beta-0.1_power_law_K0-1_m-3_exponential_lam-1.0_M4-1.csv
# model: power-law (K0=1, m=3)
# coupling: exponential (β=0.1)
# potential: exponential (lam=1.0)
# omega_m=0.143 omega_r=4.15e-05 H100=100.0 M4_tilde=1.0
# z_ini=100000000.0 phi_ini=0.0
# H0_predicted=76.6578294116729 Omega_m0_predicted=0.2433453218410334 Omega_de0_predicted=0.7565840569641662
N,a,phi,phi_prime,z,t_cosmic,eta,t_superconform,H,H_conf,Omega_m,Omega_r,Omega_de_def,w_de_def,w_phi



runs/pysco_table.txt
